# Create NER evaluation dataset

## Init

In [ ]:
import os
import sys
import subprocess  
from pathlib import Path

# 1. Environment Detection & Pre-Import Setup
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print(">>> Environment: Google Colab")
    # Install dependencies BEFORE importing them
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", 
                           "gliner2", "argilla", "tabulate", "python-dotenv", "matplotlib"])
    REPO_NAME = "archaeo-ner-greek"
    REPO_URL = f"https://github.com/prokopidis/{REPO_NAME}.git"    
    if not os.path.exists(REPO_NAME):
        os.system(f"git clone --branch dev {REPO_URL}")
    
    # Add repo to path and adjust working directory
    # Using absolute path resolution for safety
    REPO_PATH = Path(os.getcwd()) / REPO_NAME
    if str(REPO_PATH) not in sys.path:
        sys.path.append(str(REPO_PATH))
    os.chdir(str(REPO_PATH))

    # Load Colab Secrets
    from google.colab import userdata
    def get_secret(key):
        try: return userdata.get(key)
        except: return None

    env_vars = {
        "ARGILLA_API_URL": get_secret("ARGILLA_API_URL"),
        "ARGILLA_API_KEY": get_secret("ARGILLA_API_KEY"),
        "ARGILLA_WORKSPACE": get_secret("ARGILLA_WORKSPACE"),
        "ARGILLA_DATASET": get_secret("ARGILLA_DATASET"),
        "ANNOTATOR_A": get_secret("ANNOTATOR_A"),
    }
else:
    print(">>> Environment: Local")
    from dotenv import dotenv_values, find_dotenv
    env_path = find_dotenv()
    env_vars = dotenv_values(env_path) if env_path else {}

# 2. Optimized Imports (Now safe because packages are installed/pathed)
import json
import logging
import warnings
import random
from datetime import datetime
from logging.config import dictConfig

import torch
from tabulate import tabulate
from gliner2 import GLiNER2
from gliner2.training.data import InputExample, TrainingDataset
from gliner2.training.trainer import GLiNER2Trainer, TrainingConfig

# Local project imports
import archaeo_ner_greek
from archaeo_ner_greek.logging_config import LOGGING_CONFIG
from archaeo_ner_greek.utils import (
    configure_argilla_client,
    get_dataset_as_dataframe,
)

# 3. Path Management
BASE_DIR = Path(os.getcwd())
DATA_DIR = BASE_DIR / "data"
MODELS_DIR = DATA_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# 4. Logging & Global Config
dictConfig(LOGGING_CONFIG)
logger = logging.getLogger(__name__)

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings(action="ignore", message=r"datetime.datetime.utcnow")
warnings.filterwarnings("ignore", category=DeprecationWarning, message=".*SwigPyObject.*")

print(f">>> Working Directory: {BASE_DIR}")
print(f">>> Models Directory:  {MODELS_DIR}")


## Data Loading and Preprocessing


In [ ]:
DEFAULT_ANNOTATOR = env_vars.get("ANNOTATOR_A")
df_annotated = get_dataset_as_dataframe(
    client=configure_argilla_client(env_vars=env_vars),
    dataset_name=env_vars.get("ARGILLA_DATASET"),
    workspace_name=env_vars.get("ARGILLA_WORKSPACE"), 
    username=DEFAULT_ANNOTATOR
)
if not df_annotated.empty:
    logger.info(f"Ready: {len(df_annotated)} samples loaded with 'labels' ready for training.")
    logger.info(f"Available Columns: {df_annotated.columns.tolist()}")

if not df_annotated.empty:
    row = df_annotated.iloc[0]
    logger.info(f"{'='*40} FULL ROW DEBUG {'='*40}")
    logger.info(f"ID     : {row['id']}")
    logger.info(f"Full Response Dict: {json.dumps(row['sentence_field'], indent=2, ensure_ascii=False)}")
    logger.info(f"Labels (Extracted): {row['labels']}")
    logger.debug(f"Full Response Dict: {json.dumps(row['response'], indent=2, ensure_ascii=False)}")

### Guidelines to entities descriptions 

In [ ]:
# Dynamically find the package resources folder
PACKAGE_ROOT = Path(archaeo_ner_greek.__file__).parent
RESOURCES_DIR = PACKAGE_ROOT / "resources"
GUIDELINES_PATH = RESOURCES_DIR / "guidelines_en.json"
logger.info(f"Loading entity descriptions from {GUIDELINES_PATH}")
with open(GUIDELINES_PATH, 'r', encoding='utf-8') as f:
    entity_descriptions = json.load(f)

logger.info(f"Labels: {list(entity_descriptions.keys())}")
logger.info(f"Example: ARTEFACT: {entity_descriptions['ARTEFACT']}")

### Training examples

In [ ]:
train_examples = []
for _, row in df_annotated.iterrows():
    text = row['sentence_field']

    entities = {}
    for lbl in entity_descriptions.keys():
        entities[lbl] = [] # All labels are present in the InputExample, even if no instances for the label have been found

    labels = row.get('labels', [])
    for label_obj in labels:
        lbl = label_obj['label']
        start = label_obj['start']
        end = label_obj['end']
        mention = text[start:end].strip()
        # if lbl not in entities:
        #     entities[lbl] = []
        entities[lbl].append(mention) # This will crash, if the lbl is not a key in entity_descriptions.
    
    train_examples.append(InputExample(
        text=text,
        entities=entities,
        entity_descriptions=entity_descriptions,                
    ))

logger.info(f"Text: {train_examples[20].text}")
logger.info(f"Entities: {train_examples[20].entities}")
logger.info(train_examples[20].entities)
logger.info(f'Entity descriptions example: ARTEFACT: {train_examples[20].entity_descriptions["ARTEFACT"]}')
logger.info(train_examples[20])

In [ ]:
train_dataset = TrainingDataset(train_examples)
train_dataset.validate(raise_on_error=True)

# The following may throw away one example (Gliner2 bug)
# train_split, val_split, _ = train_dataset.split( 
#     train_ratio=0.9, 
#     val_ratio=0.1, 
#     test_ratio=0.0, 
#     shuffle=True, 
#     seed=42
# )

all_examples = train_dataset.examples.copy()
random.seed(42)
random.shuffle(all_examples)
val_size = int(len(all_examples) * 0.1) 
val_split = all_examples[:val_size]      
train_split = all_examples[val_size:]   
print(f"Train: {len(train_split)} | Val: {len(val_split)} | Total: {len(train_split) + len(val_split)}")
train_split = TrainingDataset(train_split)
val_split = TrainingDataset(val_split)


for ds_name, ds in {"full": train_dataset, "train": train_split, "val":val_split}.items():
    print(f"Dataset: {ds_name} ")
    ds.print_stats()
    logger.debug(ds[0])

# Training

## Custom metrics for evaluation

In [ ]:
def compute_metrics(model, dataset, threshold=0.1):
    """Bulleted Micro-F1 calculation with local path-based schema loading."""
    tp, fp, fn = 0, 0, 0
    model.eval()

    for i, ex in enumerate(dataset):
        logger.debug(ex)
        # Inference
        text = ex[0]
        gt_entities = ex[1]["entities"] # Ground truth entities
        entity_descriptions = ex[1]["entity_descriptions"]

        output = model.extract_entities(text, entity_descriptions, threshold=threshold)
        pred_entities = output.get('entities', {})
        
        # Flatten pred spans
        pred_spans = []
        for lbl, texts in pred_entities.items():
            for t in texts:
                pred_spans.append((t, lbl))
        
        # Flatten gt spans
        gt_spans = []
        for lbl, texts in gt_entities.items():
            for t in texts:
                gt_spans.append((t, lbl))
        
        # Exact Match logic (order insensitive)
        temp_gt = gt_spans.copy()
        for p in pred_spans:
            if p in temp_gt:
                tp += 1
                temp_gt.remove(p)
            else:
                fp += 1
        fn += len(temp_gt)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    metrics = {"f1": f1, "precision": precision, "recall": recall, "tp": tp, "fp": fp, "fn": fn}
    print(f"\n>>> EVAL: {metrics}")
    sys.stdout.flush()
    
    return metrics


## Training config

In [ ]:
experiment_name = f"gliner2_archaeo_lora_{datetime.now().strftime('%Y%m%d_%H%M')}"
output_dir = DATA_DIR / "models" / experiment_name
output_dir.mkdir(parents=True, exist_ok=True)
num_epochs = 30
training_config = TrainingConfig(
    output_dir=str(output_dir),
    experiment_name=experiment_name,
    seed=42,
    
    # Hardware & Batching Stability 
    batch_size=1,
    eval_batch_size=1,             # Prevents "tensor size mismatch" during evaluation
    gradient_accumulation_steps=4, # Simulates Effective Batch Size = 4
    fp16=True,                     # Half-precision for speed/memory
    
    # LoRA Architecture (Rank 4 for stability on small datasets)
    use_lora=True,
    lora_r=4,                     # Reduced from 16
    lora_alpha=8.0,               # Reduced from 32.0 (standard 2*r)
    lora_dropout=0.1,             # Regularization for small datasets
    lora_target_modules=["encoder"], # Focused target
    save_adapter_only=True,        # Saves ~10-30MB instead of 1.2GB per checkpoint

    # Optimization Profile
    num_epochs=num_epochs,
    task_lr=1e-4,                 # Primary learning rate for adapters/heads
    warmup_ratio=0.1,
    scheduler_type="cosine",       # Smooth decay for stable convergence
    weight_decay=0.01,


    # Checkpointing (Accuracy follows F1)
    eval_strategy="epoch",
    save_best=True,
    metric_for_best="f1",        # Use F1 to drive selection
    greater_is_better=True,      # Higher is better
    # metric_for_best="eval_loss", # Use Loss to drive selection
    # greater_is_better=False,      # Lower is better
    save_total_limit=2,
    logging_steps=5,
       
    # Early Stopping (DISABLED due to gliner2 v1.2.5 bug)
    early_stopping=False,
    early_stopping_patience=10,

    # Data Handling
    validate_data=True,
)

## Trainer

In [ ]:
model = GLiNER2.from_pretrained("fastino/gliner2-multi-v1") # Multi-tasking, multilingual
trainer = GLiNER2Trainer(model, training_config, compute_metrics=compute_metrics)

## Train run

In [ ]:
results = trainer.train(
    train_data=train_split, 
    eval_data=val_split
)

## Dev evaluation 

In [ ]:
best_run = max(results["eval_metrics_history"], key=lambda x: x['f1'])
best_epoch = best_run['epoch']
best_p = best_run['precision']
best_r = best_run['recall']
best_f1 = best_run['f1']
total_epochs = len(results["eval_metrics_history"])
def get_cnt(data):
    exs = getattr(data, "examples", data)
    return sum(len(mentions) for ex in exs for mentions in ex.entities.values())

print(f"Training completed!")
print(f"Experiment name: {experiment_name}")
print(f"Total steps: {results['total_steps']}")
print(f"Total epochs: {total_epochs}")
print(f"Training time: {results['total_time_seconds']/60:.1f} minutes")
print(f"Training completed!")
print(f"Best Epoch: {best_epoch + 1}/{total_epochs}") # +1 for 1-based indexing
print(f"Best PRF: Precision: {best_p:.4f}, Recall: {best_r:.4f}, F1: {best_f1:.4f}")
# 2. Prepare data rows
table_data = [
    ["Train Split",  len(train_split),             get_cnt(train_split)],
    ["Val Split",    len(val_split),               get_cnt(val_split)],
    ["Full Dataset", len(train_dataset.examples), get_cnt(train_dataset)]
]
# 3. Print table
print(tabulate(table_data, headers=["Subset", "Samples", "Mentions"], tablefmt="rounded_grid"))


In [ ]:
import matplotlib.pyplot as plt

# 1. Extract metrics from history
history = results['eval_metrics_history']
epochs = [h['epoch'] + 1 for h in history]
f1_scores = [h['f1'] for h in history]
precision = [h['precision'] for h in history]
recall = [h['recall'] for h in history]
losses = [h['eval_loss'] for h in history]

# 2. Setup the plot
plt.figure(figsize=(12, 5))

# Plot 1: PRF Metrics
plt.subplot(1, 2, 1)
plt.plot(epochs, f1_scores, label='F1', marker='o', color='#1f77b4', linewidth=2)
plt.plot(epochs, precision, label='Precision', linestyle='--', alpha=0.7)
plt.plot(epochs, recall, label='Recall', linestyle='--', alpha=0.7)
plt.title(f"Model Performance: {experiment_name}")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.grid(True, alpha=0.3)
plt.legend()

# Plot 2: Evaluation Loss
plt.subplot(1, 2, 2)
plt.plot(epochs, losses, label='Eval Loss', color='#d62728', marker='s')
plt.title("Convergence (Loss)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()


# 3. Performance Analysis
Quantitative and qualitative assessment of the model's performance.
- **Metrics**: Precision, Recall, and F1-score.
- **Error Analysis**: Review of overlapping spans and MISC classification suggestions.


In [ ]:

# 1. Load the original base model (pristine weights)
best_model = GLiNER2.from_pretrained("fastino/gliner2-multi-v1")
# 2. Add the LoRA
adapter_path = DATA_DIR / "models" / experiment_name / "best"
best_model.load_adapter(adapter_path)
# 3. Ready for inference
print("Adapter loaded.")


def evaluate_adapter(model, adapter_path, test_data, threshold=0.1):
    """
    Loads a specific LoRA adapter and evaluates its performance.
    """
    # 1. Load the specific weights
    print(f"Loading adapter from: {adapter_path}")
    model.load_adapter(adapter_path)
    
    test_data = [
        (ex.text, {"entities": ex.entities, "entity_descriptions": ex.entity_descriptions}) 
        for ex in test_data
    ]

    # 2. Execute metric calculation
    results = compute_metrics(model, test_data, threshold=threshold)
    print(f"\n--- EVALUATION RESULTS ({adapter_path.name}) threshold: {threshold} ---")
    print(f"F1 Score : {results['f1']}")
    print(f"Precision: {results['precision']}")
    print(f"Recall   : {results['recall']}")
    print(f"Counts   : TP={results['tp']}, FP={results['fp']}, FN={results['fn']}")
    
    return results

final_results = evaluate_adapter(best_model, adapter_path, val_split, threshold=0.1)
